# Adaptive Learning Platform - Phase 1

This notebook prototypes accessible text transformation and quiz generation before API wiring. Run the cells from top to bottom.

## 1. SETUP

Imports, environment configuration, the shared LLM wrapper, and sample input.

In [1]:
import json
import re
import os
import io
import base64
import textwrap
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
load_dotenv()

import pdfplumber
import requests
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe in Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

try:
    from groq import Groq
except ImportError:
    Groq = None

# ── ORIGINAL PIPELINE KEY (unchanged) ─────────────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_KEY")
GROQ_MODEL   = os.getenv("GROQ_MODEL")
GROQ_CLIENT  = None

if GROQ_API_KEY and Groq is not None:
    GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
    if not GROQ_MODEL:
        available_models = {m.id for m in GROQ_CLIENT.models.list().data}
        preferred_models = (
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        )
        GROQ_MODEL = next(
            (m for m in preferred_models if m in available_models),
            next((m for m in available_models if "llama" in m or "gpt" in m), None),
        )
    if not GROQ_MODEL:
        raise RuntimeError("No text-generation model available. Set GROQ_MODEL in .env.")


def call_llm(prompt: str) -> str:
    """Generate text with the ORIGINAL Groq key, or use the local fallback."""
    if GROQ_API_KEY:
        if GROQ_CLIENT is None:
            raise ImportError("Install the groq package before using GROQ_API_KEY.")
        opts = {
            "model": GROQ_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
            "max_completion_tokens": 4096,
        }
        if "QUIZ_JSON" in prompt or "CHUNK_JSON" in prompt:
            opts["response_format"] = {"type": "json_object"}
        if GROQ_MODEL.startswith("openai/"):
            opts["reasoning_effort"] = "low"
        return GROQ_CLIENT.chat.completions.create(**opts).choices[0].message.content or ""

    if "QUIZ_JSON" in prompt:
        return json.dumps({
            "question": "What do plants use photosynthesis to produce?",
            "options": ["Glucose", "Sound", "Salt", "Metal"],
            "answer": "Glucose",
            "explanation": "Photosynthesis produces glucose, which stores chemical energy.",
        })
    if "CHUNK_JSON" in prompt:
        sentences = re.split(r"(?<=[.!?])\s+", prompt.split("TEXT:", 1)[-1].strip())
        return json.dumps({"chunks": [" ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]})
    return (
        "Photosynthesis lets plants make food from sunlight. Chlorophyll captures light energy. "
        "Plants use water and carbon dioxide to produce glucose and release oxygen."
    )


print("Groq SDK available:", Groq is not None)
print("Groq API configured:", bool(GROQ_API_KEY))
print("Groq model:", GROQ_MODEL or "local fallback")

# ── VOICE KEY (separate) ───────────────────────────────────────────────────────
VOICE_GROQ_API_KEY = os.getenv("VOICE_GROQ_API_KEY")
VOICE_GROQ_CLIENT  = None
VOICE_GROQ_MODEL   = None

if VOICE_GROQ_API_KEY and Groq is not None:
    VOICE_GROQ_CLIENT = Groq(api_key=VOICE_GROQ_API_KEY)
    try:
        _vm = {m.id for m in VOICE_GROQ_CLIENT.models.list().data}
        _vp = ("llama-3.3-70b-versatile", "llama-3.1-8b-instant", "openai/gpt-oss-20b")
        VOICE_GROQ_MODEL = next((m for m in _vp if m in _vm),
                                next((m for m in _vm if "llama" in m or "gpt" in m), None))
    except Exception:
        VOICE_GROQ_MODEL = None

print("Voice key configured:", bool(VOICE_GROQ_API_KEY))

# ── VISUAL KEY (separate) ──────────────────────────────────────────────────────
VISUAL_GROQ_API_KEY = os.getenv("VISUAL_GROQ_API_KEY")
VISUAL_GROQ_CLIENT  = None
VISUAL_GROQ_MODEL   = None

if VISUAL_GROQ_API_KEY and Groq is not None:
    VISUAL_GROQ_CLIENT = Groq(api_key=VISUAL_GROQ_API_KEY)
    try:
        _avm = {m.id for m in VISUAL_GROQ_CLIENT.models.list().data}
        _avp = ("llama-3.3-70b-versatile", "llama-3.1-8b-instant", "openai/gpt-oss-20b")
        VISUAL_GROQ_MODEL = next((m for m in _avp if m in _avm),
                                 next((m for m in _avm if "llama" in m or "gpt" in m), None))
    except Exception:
        VISUAL_GROQ_MODEL = None

print("Visual key configured:", bool(VISUAL_GROQ_API_KEY))

# ── VOICE HELP ─────────────────────────────────────────────────────────────────
VOICE_HELP_PROMPT = """You are an educational assistant inside an adaptive learning platform.
Answer the learner's request using ONLY the supplied lesson content.
Do not invent facts.
Explain clearly and simply.
Keep the response concise (3-5 sentences where possible).
If the answer cannot be determined from the lesson, say:
  \"The lesson does not provide enough information to answer that.\"

LESSON CONTENT:
{lesson}

LEARNER REQUEST:
{request}"""


def voice_ask(user_request: str, lesson_text: str = None) -> str:
    """Voice Q&A — uses VOICE_GROQ_API_KEY when set, falls back to main key."""
    context = (lesson_text or "").strip()
    if not context:
        try:
            context = SOURCE_TEXT  # noqa: F821
        except NameError:
            context = "[Lesson text not yet loaded.]"

    prompt = VOICE_HELP_PROMPT.format(lesson=context, request=user_request.strip())
    try:
        # Prefer the dedicated voice key; fall back to the main call_llm
        if VOICE_GROQ_CLIENT and VOICE_GROQ_MODEL:
            opts = {
                "model": VOICE_GROQ_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.2,
                "max_completion_tokens": 1024,
            }
            if VOICE_GROQ_MODEL.startswith("openai/"):
                opts["reasoning_effort"] = "low"
            return VOICE_GROQ_CLIENT.chat.completions.create(**opts).choices[0].message.content or ""
        return call_llm(prompt)
    except Exception as exc:
        return f"[PRISM error: {exc}]"


# ── VISUAL LEARNING ────────────────────────────────────────────────────────────
VISUAL_SPEC_PROMPT = """You are an educational visual designer inside an adaptive learning platform.
Analyse the lesson below and decide whether a visual representation would help the learner.

PROFILE: {profile}
Profile-specific rules:
- dyslexia: minimal text, very short labels (2-4 words), wide spacing, simple structure, max 6 nodes.
- cognitive_load: step-by-step structure, max 5 nodes, no clutter, one idea per node.
- low_vision: large readable labels, strong contrast colours, larger visual elements, max 7 nodes.

LESSON:
{lesson}

Return STRICT JSON only. No markdown fences. No extra keys. Exactly this structure:
{{
  \"should_visualize\": true,
  \"visual_type\": \"flowchart\",
  \"title\": \"...\",
  \"description\": \"...\",
  \"why_helpful\": \"1-2 sentence explanation of why this visual helps the learner.\",
  \"nodes\": [\"Step 1\", \"Step 2\"],
  \"edges\": [[0,1]],
  \"labels\": [\"label on edge 0->1\"],
  \"data\": []
}}

Rules:
- visual_type must be one of: flowchart, timeline, process, graph, concept_map, none.
- If visual_type is \"none\", set should_visualize to false and leave nodes/edges/labels/data empty.
- Use ONLY facts from the lesson. Do not invent anything.
- nodes: array of short label strings.
- edges: array of [from_index, to_index] pairs (integers).
- labels: array of strings for edge labels (same length as edges, use \"\" for unlabelled edges).
- data: for visual_type \"graph\" only — array of {{\"label\":\"...\",\"value\":number}} objects.
  For all other types, data must be [].
- Keep node labels concise. Profile dyslexia/cognitive_load: max 4 words per label."""


def _call_visual_llm(prompt: str) -> str:
    """Call Groq with VISUAL_GROQ_API_KEY. Falls back to main key if not set."""
    if VISUAL_GROQ_CLIENT and VISUAL_GROQ_MODEL:
        opts = {
            "model": VISUAL_GROQ_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_completion_tokens": 2048,
            "response_format": {"type": "json_object"},
        }
        if VISUAL_GROQ_MODEL.startswith("openai/"):
            opts["reasoning_effort"] = "low"
        return VISUAL_GROQ_CLIENT.chat.completions.create(**opts).choices[0].message.content or ""
    # Fallback: main key — add json_object format hint through the existing call_llm path
    return call_llm(prompt)


_VISUAL_SPEC_REQUIRED_KEYS = {"should_visualize", "visual_type", "title",
                               "description", "why_helpful", "nodes",
                               "edges", "labels", "data"}
_VALID_VISUAL_TYPES = {"flowchart", "timeline", "process", "graph", "concept_map", "none"}


def generate_visual_spec(lesson_text: str, profile: str) -> Dict[str, Any]:
    """Ask Groq to produce a visual specification for the given lesson + profile.

    Returns a validated dict. Never raises — returns an error dict on failure.
    """
    if not VISUAL_GROQ_API_KEY and not GROQ_API_KEY:
        return {
            "error": "No Groq API key is configured. "
                     "Set VISUAL_GROQ_API_KEY (or GROQ_KEY) in your .env file."
        }

    prompt = VISUAL_SPEC_PROMPT.format(
        profile=profile,
        lesson=lesson_text[:6000],   # cap to stay within token budget
    )
    try:
        raw = _call_visual_llm(prompt)
        cleaned = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "",
                         raw.strip(), flags=re.IGNORECASE)
        spec = json.loads(cleaned)
    except Exception as exc:
        return {"error": f"Groq returned invalid JSON: {exc}"}

    # Validate required keys
    missing = _VISUAL_SPEC_REQUIRED_KEYS - set(spec.keys())
    if missing:
        return {"error": f"Visual spec missing keys: {missing}"}
    if spec.get("visual_type") not in _VALID_VISUAL_TYPES:
        return {"error": f"Unknown visual_type: {spec.get('visual_type')}"}

    # Ensure edge indices are valid
    n = len(spec.get("nodes", []))
    for edge in spec.get("edges", []):
        if not (isinstance(edge, list) and len(edge) == 2
                and 0 <= edge[0] < n and 0 <= edge[1] < n):
            return {"error": f"Invalid edge in spec: {edge} (nodes length={n})"}

    return spec


# ── RENDER VISUAL SPEC ─────────────────────────────────────────────────────────
def _fig_to_base64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=130)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


def _profile_style(profile: str) -> Dict[str, Any]:
    """Return matplotlib style parameters keyed to the accessibility profile."""
    if profile == "dyslexia":
        return dict(fontsize=13, node_color="#dbeafe", edge_color="#1d4ed8",
                    text_color="#1e3a5f", figsize=(9, 5), lw=2.5)
    if profile == "cognitive_load":
        return dict(fontsize=12, node_color="#dcfce7", edge_color="#166534",
                    text_color="#14532d", figsize=(9, 5), lw=2)
    # low_vision
    return dict(fontsize=16, node_color="#fef08a", edge_color="#713f12",
                text_color="#000000", figsize=(11, 6), lw=3)


def _render_linear(spec: Dict[str, Any], profile: str,
                   direction: str = "vertical") -> str:
    """Render a flowchart / timeline / process as a linear chain of boxes."""
    nodes  = spec["nodes"]
    labels = spec.get("labels", [])
    st     = _profile_style(profile)
    n      = len(nodes)
    if n == 0:
        return ""

    is_horiz = (direction == "horizontal")
    fw, fh   = (st["figsize"][1] * 1.8, st["figsize"][1]) if is_horiz else st["figsize"]
    fig, ax  = plt.subplots(figsize=(fw, fh))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    bw = 0.55 if is_horiz else 0.60
    bh = 0.10 if is_horiz else 0.09
    gap = (1.0 - bh * n) / (n + 1) if not is_horiz else (1.0 - bw * n) / (n + 1)

    cx_base = 0.50
    cy_base = 1.0

    centers = []
    for i, label in enumerate(nodes):
        if is_horiz:
            cx = gap + bw * i + gap * i + bw / 2
            cy = 0.50
        else:
            cx = cx_base
            cy = cy_base - gap - bh * i - gap * i - bh / 2
        centers.append((cx, cy))

        # Wrap long labels
        wrapped = "\n".join(textwrap.wrap(label, width=22))
        rect = mpatches.FancyBboxPatch(
            (cx - bw / 2, cy - bh / 2), bw, bh,
            boxstyle="round,pad=0.02",
            linewidth=st["lw"], edgecolor=st["edge_color"],
            facecolor=st["node_color"], zorder=3,
        )
        ax.add_patch(rect)
        ax.text(cx, cy, wrapped, ha="center", va="center",
                fontsize=st["fontsize"], color=st["text_color"],
                fontweight="bold", zorder=4, wrap=True)

    # Draw arrows between consecutive nodes
    for i in range(n - 1):
        x0, y0 = centers[i]
        x1, y1 = centers[i + 1]
        if is_horiz:
            ax.annotate("", xy=(x1 - bw / 2, y1),
                        xytext=(x0 + bw / 2, y0),
                        arrowprops=dict(arrowstyle="->", color=st["edge_color"],
                                        lw=st["lw"]))
        else:
            ax.annotate("", xy=(x1, y1 + bh / 2),
                        xytext=(x0, y0 - bh / 2),
                        arrowprops=dict(arrowstyle="->", color=st["edge_color"],
                                        lw=st["lw"]))
        # Edge label
        if i < len(labels) and labels[i]:
            mx = (x0 + x1) / 2 + (0.04 if not is_horiz else 0)
            my = (y0 + y1) / 2
            ax.text(mx, my, labels[i], ha="left", va="center",
                    fontsize=st["fontsize"] - 2, color=st["edge_color"], style="italic")

    ax.set_title(spec.get("title", ""), fontsize=st["fontsize"] + 2,
                 fontweight="bold", color=st["text_color"], pad=10)
    return _fig_to_base64(fig)


def _render_graph(spec: Dict[str, Any], profile: str) -> str:
    """Render a bar chart for numerical data."""
    data = spec.get("data", [])
    if not data:
        return ""
    st = _profile_style(profile)
    labels_g = [d.get("label", str(i)) for i, d in enumerate(data)]
    values   = [float(d.get("value", 0)) for d in data]

    fig, ax = plt.subplots(figsize=st["figsize"])
    bars = ax.bar(labels_g, values, color=st["node_color"],
                  edgecolor=st["edge_color"], linewidth=st["lw"])
    ax.set_title(spec.get("title", ""), fontsize=st["fontsize"] + 2,
                 fontweight="bold", color=st["text_color"])
    ax.tick_params(labelsize=st["fontsize"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                str(val), ha="center", va="bottom",
                fontsize=st["fontsize"] - 1, color=st["text_color"])
    plt.tight_layout()
    return _fig_to_base64(fig)


def _render_concept_map(spec: Dict[str, Any], profile: str) -> str:
    """Render a concept map as a hub-and-spoke layout."""
    nodes  = spec["nodes"]
    edges  = spec.get("edges", [])
    labels = spec.get("labels", [])
    st     = _profile_style(profile)
    n      = len(nodes)
    if n == 0:
        return ""

    fig, ax = plt.subplots(figsize=(st["figsize"][0] + 2, st["figsize"][1] + 1))
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.4, 1.4)
    ax.axis("off")

    # Position nodes: first node at centre, rest around a circle
    positions = []
    if n == 1:
        positions = [(0, 0)]
    else:
        positions = [(0, 0)]
        ring_n = n - 1
        for k in range(ring_n):
            angle = 2 * np.pi * k / ring_n - np.pi / 2
            positions.append((1.1 * np.cos(angle), 1.0 * np.sin(angle)))

    bw, bh = 0.55, 0.16
    for i, (lbl, (px, py)) in enumerate(zip(nodes, positions)):
        wrapped = "\n".join(textwrap.wrap(lbl, width=18))
        rect = mpatches.FancyBboxPatch(
            (px - bw / 2, py - bh / 2), bw, bh,
            boxstyle="round,pad=0.03",
            linewidth=st["lw"], edgecolor=st["edge_color"],
            facecolor=st["node_color"], zorder=3,
        )
        ax.add_patch(rect)
        ax.text(px, py, wrapped, ha="center", va="center",
                fontsize=st["fontsize"] - 1, color=st["text_color"],
                fontweight="bold", zorder=4)

    for idx, (ei, ej) in enumerate(edges):
        if ei >= n or ej >= n:
            continue
        x0, y0 = positions[ei]
        x1, y1 = positions[ej]
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle="->", color=st["edge_color"],
                                   lw=st["lw"],
                                   connectionstyle="arc3,rad=0.1"),
                    zorder=2)
        if idx < len(labels) and labels[idx]:
            mx, my = (x0 + x1) / 2, (y0 + y1) / 2
            ax.text(mx, my, labels[idx], ha="center", va="bottom",
                    fontsize=st["fontsize"] - 3, color=st["edge_color"], style="italic")

    ax.set_title(spec.get("title", ""), fontsize=st["fontsize"] + 2,
                 fontweight="bold", color=st["text_color"], pad=10)
    return _fig_to_base64(fig)


def render_visual_spec(spec: Dict[str, Any], profile: str) -> Optional[str]:
    """Convert a validated visual spec into a base64-encoded PNG.

    Returns None when the spec says no visual is needed or rendering fails.
    """
    if "error" in spec or not spec.get("should_visualize", False):
        return None
    vtype = spec.get("visual_type", "none")
    try:
        if vtype in ("flowchart", "process"):
            return _render_linear(spec, profile, direction="vertical")
        if vtype == "timeline":
            return _render_linear(spec, profile, direction="horizontal")
        if vtype == "graph":
            return _render_graph(spec, profile)
        if vtype == "concept_map":
            return _render_concept_map(spec, profile)
    except Exception as exc:
        print(f"[render_visual_spec] rendering error: {exc}")
    return None


print("Visual learning helpers loaded.")

Groq SDK available: True
Groq API configured: True
Groq model: openai/gpt-oss-20b


Voice key configured: True


Visual key configured: True
Visual learning helpers loaded.


## 2. PDF EXTRACTION

PDF text extraction uses `pdfplumber`, with a page-level fallback for image-only or otherwise empty pages.

In [2]:
def extract_text_and_tables_from_pdf(filepath: str) -> tuple[str, List[List[List[str]]]]:
    """Extract prose text and tables separately so table rows don't bleed into paragraphs."""
    pages_text:   List[str]                    = []
    pages_tables: List[List[List[List[str]]]]  = []

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                found_tables = page.find_tables()
                table_bboxes = [t.bbox for t in found_tables]

                def is_inside_a_table(obj, boxes=table_bboxes) -> bool:
                    for (tx0, ttop, tx1, tbottom) in boxes:
                        if obj["x0"] >= tx0 and obj["x1"] <= tx1 and obj["top"] >= ttop and obj["bottom"] <= tbottom:
                            return True
                    return False

                prose_only_page = page.filter(lambda obj: not is_inside_a_table(obj))
                page_text = (prose_only_page.extract_text() or "").strip()
                if not page_text:
                    page_text = f"[No extractable prose text found on page {page_number}; OCR may be required.]"
                pages_text.append(page_text)
                pages_tables.append([t.extract() for t in found_tables])
    except FileNotFoundError:
        raise FileNotFoundError(f"PDF file not found: {filepath}")
    except Exception as error:
        raise RuntimeError(f"Could not extract PDF content from {filepath}: {error}") from error

    return "\n\n".join(pages_text), pages_tables


PDF_PATH = "./sample_input.pdf"
SOURCE_TEXT, SOURCE_TABLES = extract_text_and_tables_from_pdf(PDF_PATH)

print("--- Extracted prose text ---")
print(SOURCE_TEXT)

print("\n--- Extracted tables ---")
for page_num, page_tables in enumerate(SOURCE_TABLES, start=1):
    for table_num, table in enumerate(page_tables, start=1):
        print(f"\nPage {page_num}, table {table_num}:")
        for row in table:
            print(row)

print("\nSource words:", len(SOURCE_TEXT.split()))
print("Tables found:", sum(len(t) for t in SOURCE_TABLES))

--- Extracted prose text ---


Chapter 7: Photosynthesis and Energy Flow
Grade 9 Biology · Unit 3: Plant Systems
7.1 What is Photosynthesis?
Photosynthesis is the biochemical process by which green plants, algae, and certain bacteria
convert light energy, typically from the sun, into chemical energy stored in glucose molecules.
This process is fundamental to almost all life on Earth, as it forms the base of most food chains
and is responsible for producing the oxygen that most organisms depend on for cellular
respiration. The overall chemical equation for photosynthesis can be summarized as carbon
dioxide plus water, in the presence of light energy, yielding glucose and oxygen.
The process takes place primarily in the chloroplasts of plant cells, specialized organelles that
contain a green pigment called chlorophyll. Chlorophyll is essential because it absorbs light most
efficiently in the blue and red wavelengths of the visible spectrum, while reflecting green light,
which is why most plants appear green to the hu

## 3. PROFILE-BASED TRANSFORMATION

Prompt constants are deliberately separate from the transformation function so they can be edited without changing control flow.

In [3]:
DYSLEXIA_PROMPT = """Rewrite the text below for a reader with dyslexia. Use short, simple sentences and clear wording. Preserve every fact, relationship, number, and cause-and-effect detail. Do not add facts. Return only the rewritten text.

TEXT:
{text}"""

COGNITIVE_LOAD_PROMPT = """Split the text below into an ordered JSON object with one key, chunks. The chunks value must be an array of digestible chunks. Each chunk must contain 2 to 4 complete sentences. Preserve all original content and facts, do not summarize, and do not add facts. Return only valid JSON. Include the marker CHUNK_JSON nowhere except in this instruction context.

TEXT:
{text}"""

def _parse_json_response(response: str) -> Any:
    """Remove optional Markdown fences and parse a JSON response."""
    cleaned = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", response.strip(), flags=re.IGNORECASE)
    return json.loads(cleaned)

def transform_text(text: str, profile: str) -> Dict[str, Any]:
    """Transform text according to an accessibility profile."""
    if profile == "low_vision":
        return {"profile": profile, "text": text, "formatting": {"font_size_multiplier": 1.5, "contrast_mode": "high"}}
    if profile == "dyslexia":
        rewritten = call_llm(DYSLEXIA_PROMPT.format(text=text))
        if not rewritten.strip():
            raise ValueError("Groq returned an empty dyslexia transformation.")
        return {"profile": profile, "text": rewritten.strip()}
    if profile == "cognitive_load":
        response = call_llm(COGNITIVE_LOAD_PROMPT.format(text=text))
        parsed = _parse_json_response(response)
        chunks = parsed.get("chunks") if isinstance(parsed, dict) else parsed
        if not isinstance(chunks, list) or not all(isinstance(c, str) and c.strip() for c in chunks):
            raise ValueError("Cognitive-load response must contain a JSON chunks array of strings.")
        return {"profile": profile, "chunks": chunks}
    raise ValueError("profile must be dyslexia, low_vision, or cognitive_load")

for profile in ("dyslexia", "low_vision", "cognitive_load"):
    print(f"\n--- {profile.upper()} ---")
    print(json.dumps(transform_text(SOURCE_TEXT, profile), indent=2, ensure_ascii=False))


--- DYSLEXIA ---


{
  "profile": "dyslexia",
  "text": "Chapter 7: Photosynthesis and Energy Flow  \nGrade 9 Biology · Unit 3: Plant Systems  \n\n7.1 What is Photosynthesis?  \nPhotosynthesis is a chemical process.  \nGreen plants, algae, and some bacteria do it.  \nThey use light energy, usually from the sun.  \nThey turn that light into chemical energy.  \nThe energy is stored in glucose molecules.  \nThis process is very important.  \nIt is the base of most food chains.  \nIt makes oxygen.  \nOxygen is needed for cellular respiration.  \nThe overall equation is:  \ncarbon dioxide + water + light → glucose + oxygen.  \n\nThe process happens mainly in chloroplasts.  \nChloroplasts are special parts of plant cells.  \nThey contain a green pigment called chlorophyll.  \nChlorophyll absorbs light best in blue and red wavelengths.  \nIt reflects green light.  \nThat is why plants look green.  \n\nInside the chloroplast, photosynthesis has two stages.  \nEach stage happens in a different place.  \nEach stag

{
  "profile": "cognitive_load",
  "chunks": [
    "Photosynthesis is the biochemical process by which green plants, algae, and certain bacteria convert light energy, typically from the sun, into chemical energy stored in glucose molecules. This process is fundamental to almost all life on Earth, as it forms the base of most food chains and is responsible for producing the oxygen that most organisms depend on for cellular respiration. The overall chemical equation for photosynthesis can be summarized as carbon dioxide plus water, in the presence of light energy, yielding glucose and oxygen.",
    "The process takes place primarily in the chloroplasts of plant cells, specialized organelles that contain a green pigment called chlorophyll. Chlorophyll is essential because it absorbs light most efficiently in the blue and red wavelengths of the visible spectrum, while reflecting green light, which is why most plants appear green to the human eye. Within the chloroplast, photosynthesis unfo

In [4]:
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML

PROFILE_LABELS = {
    "I have dyslexia": "dyslexia",
    "I find long or dense text difficult": "cognitive_load",
    "I need larger, high-contrast text": "low_vision",
}

# ── EXISTING WIDGETS (unchanged) ───────────────────────────────────────────────
profile_selector = widgets.Dropdown(
    options=list(PROFILE_LABELS),
    value="I have dyslexia",
    description="I am dealing with:",
    layout=widgets.Layout(width="550px"),
)
text_input = widgets.Textarea(
    value=SOURCE_TEXT,
    description="Text:",
    layout=widgets.Layout(width="700px", height="180px"),
)
generate_button     = widgets.Button(description="Generate accessible version", button_style="primary")
transformation_output = widgets.Output()

def render_transformation(result: Dict[str, Any]) -> None:
    with transformation_output:
        transformation_output.clear_output()
        if result["profile"] == "cognitive_load":
            print("Generated digestible chunks:\n")
            for number, chunk in enumerate(result["chunks"], start=1):
                print(f"{number}. {chunk}\n")
        else:
            print(result["text"])
            if result["profile"] == "low_vision":
                print("\nDisplay settings: larger text (1.5x), high contrast")

def generate_selected_transformation(_button):
    text = text_input.value.strip()
    with transformation_output:
        transformation_output.clear_output()
        if not text:
            print("Please enter some text before generating an accessible version.")
            return
        try:
            selected_profile = PROFILE_LABELS[profile_selector.value]
            render_transformation(transform_text(text, selected_profile))
        except (ValueError, json.JSONDecodeError) as error:
            print(f"Could not generate the transformation: {error}")

generate_button.on_click(generate_selected_transformation)

# ── VOICE SECTION (unchanged) ──────────────────────────────────────────────────
_voice_bridge = widgets.Text(value="", layout=widgets.Layout(display="none"))
_voice_bridge.add_class("prism-voice-bridge")

voice_text_input = widgets.Text(
    placeholder="Or type your question here…",
    layout=widgets.Layout(width="600px"),
)
voice_text_input.add_class("prism-voice-input")

mic_button = widgets.Button(description="🎙️ Speak", button_style="info",
                            layout=widgets.Layout(width="130px"),
                            tooltip="Click and speak (Chrome/Edge only)")
mic_button.add_class("prism-mic-btn")

ask_button = widgets.Button(description="💬 Ask", button_style="warning",
                            layout=widgets.Layout(width="100px"))
ask_button.add_class("prism-ask-btn")

read_aloud_button = widgets.Button(description="🔊 Read Aloud", button_style="success",
                                   layout=widgets.Layout(width="140px", display="none"))
read_aloud_button.add_class("prism-read-aloud-btn")

voice_output = widgets.Output()

VOICE_JS_CODE = """
(function () {
  function setVoiceStatus(html) {
    var el = document.getElementById('prism-voice-status');
    if (el) el.innerHTML = html;
  }
  function getBridgeInput() {
    var el = document.querySelector('.prism-voice-bridge input');
    if (el) return el;
    var inputs = document.querySelectorAll('.widget-text input');
    for (var i = 0; i < inputs.length; i++) {
      var w = inputs[i].closest('.widget-text');
      if (w && (w.style.display === 'none' || w.hidden)) return inputs[i];
    }
    return null;
  }
  function getVisibleInput() {
    var el = document.querySelector('.prism-voice-input input');
    if (el) return el;
    var inputs = document.querySelectorAll('.widget-text input');
    for (var i = 0; i < inputs.length; i++) {
      var w = inputs[i].closest('.widget-text');
      if (w && w.style.display !== 'none' && !w.hidden) return inputs[i];
    }
    return null;
  }
  window.startVoiceRecognition = function () {
    var SR = window.SpeechRecognition || window.webkitSpeechRecognition;
    if (!SR) {
      setVoiceStatus('<span style="color:#ef4444;">Speech recognition not supported. Use Chrome/Edge or type below.</span>');
      return;
    }
    setVoiceStatus('<span style="color:#10b981;font-weight:bold;">🎙️ Listening…</span>');
    var recog = new SR();
    recog.lang = 'en-US'; recog.interimResults = false; recog.maxAlternatives = 1;
    recog.onresult = function(e) {
      var t = e.results[0][0].transcript;
      setVoiceStatus('<span style="color:#0284c7;font-weight:bold;">Heard: "' + t + '"</span>');
      var vis = getVisibleInput();
      if (vis) {
        var s = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype,'value').set;
        s.call(vis, t); vis.dispatchEvent(new Event('input',{bubbles:true}));
        vis.dispatchEvent(new Event('change',{bubbles:true}));
      }
      var br = getBridgeInput();
      if (br) {
        var s2 = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype,'value').set;
        s2.call(br, t); br.dispatchEvent(new Event('input',{bubbles:true}));
        br.dispatchEvent(new Event('change',{bubbles:true}));
      } else {
        var ab = document.querySelector('.prism-ask-btn'); if (ab) ab.click();
      }
    };
    recog.onerror = function(e) {
      setVoiceStatus(e.error==='not-allowed'
        ? '<span style="color:#ef4444;">❌ Mic blocked. Allow in browser URL bar.</span>'
        : '<span style="color:#f59e0b;">Error: '+e.error+'. Try again or type.</span>');
    };
    recog.onend = function() {
      var el=document.getElementById('prism-voice-status');
      if (el && el.innerText.includes('Listening'))
        setVoiceStatus('No speech detected. Try again or type.');
    };
    try { recog.start(); } catch(err) {
      setVoiceStatus('<span style="color:#ef4444;">Mic error: '+err.message+'</span>');
    }
  };
  window.prismReadAloud = function(text) {
    if (!window.speechSynthesis) {
      setVoiceStatus('Speech synthesis not supported.'); return;
    }
    speechSynthesis.cancel();
    var u = new SpeechSynthesisUtterance(text);
    u.lang='en-US'; u.rate=0.95; speechSynthesis.speak(u);
  };
  function attachDomListeners() {
    document.querySelectorAll('button').forEach(function(b) {
      if (b.innerText && b.innerText.includes('Speak'))
        b.onclick = function(e){e.preventDefault();e.stopPropagation();window.startVoiceRecognition();};
      if (b.innerText && b.innerText.includes('Read Aloud'))
        b.onclick = function(e){e.preventDefault();e.stopPropagation();
          if(window._lastAiSpeech) window.prismReadAloud(window._lastAiSpeech);};
    });
  }
  attachDomListeners();
  [500,1500,3000].forEach(function(d){setTimeout(attachDomListeners,d);});
})();
"""

_last_ai_response = [""]

def _run_voice_ask(question: str) -> None:
    question = question.strip()
    if not question:
        with voice_output:
            voice_output.clear_output()
            print("Please speak or type a question first.")
        return
    with voice_output:
        voice_output.clear_output()
        print(f'You said: "{question}"\n')
        print("PRISM is thinking…")
    lesson   = text_input.value.strip() or SOURCE_TEXT
    response = voice_ask(question, lesson_text=lesson)
    _last_ai_response[0] = response
    safe_resp = response.replace("\\", "\\\\").replace("`", "\\`").replace("$", "\\$")
    with voice_output:
        voice_output.clear_output()
        print(f'You said: "{question}"\n')
        print(f"PRISM:\n{response}")
        display(Javascript(f"window._lastAiSpeech = `{safe_resp}`;"))
    read_aloud_button.layout.display = ""

def on_mic_button_click(_btn):
    with voice_output:
        display(Javascript("window.startVoiceRecognition && window.startVoiceRecognition();"))

def on_ask_button_click(_btn):
    question = _voice_bridge.value.strip() or voice_text_input.value.strip()
    _run_voice_ask(question)

def on_read_aloud_click(_btn):
    text = _last_ai_response[0]
    if not text:
        return
    safe = text.replace("\\", "\\\\").replace("`", "\\`").replace("$", "\\$")
    with voice_output:
        display(Javascript(f"window.prismReadAloud && window.prismReadAloud(`{safe}`);"))

def _on_bridge_change(change):
    new_val = change["new"].strip()
    if new_val:
        voice_text_input.value = new_val
        _run_voice_ask(new_val)

_voice_bridge.observe(_on_bridge_change, names="value")
mic_button.on_click(on_mic_button_click)
ask_button.on_click(on_ask_button_click)
read_aloud_button.on_click(on_read_aloud_click)

# ── VISUAL SECTION (new) ───────────────────────────────────────────────────────
visual_generate_button = widgets.Button(
    description="📊 Generate Visual",
    button_style="primary",
    layout=widgets.Layout(width="180px"),
    tooltip="Generate a visual explanation of the current lesson for the selected profile",
)
visual_output = widgets.Output()

def on_generate_visual(_btn):
    """Generate and display a visual spec for the current lesson + profile."""
    lesson  = text_input.value.strip() or SOURCE_TEXT
    profile = PROFILE_LABELS[profile_selector.value]

    with visual_output:
        visual_output.clear_output(wait=True)
        display(HTML("<p style='color:#6b7280;font-style:italic;'>Analysing lesson for visual opportunities…</p>"))

    spec = generate_visual_spec(lesson, profile)

    with visual_output:
        visual_output.clear_output(wait=True)

        # ── Error from key missing or bad JSON ────────────────────────────────
        if "error" in spec:
            display(HTML(
                f"<div style='padding:12px;background:#fef2f2;border:1px solid #fca5a5;"
                f"border-radius:8px;color:#991b1b;'>"
                f"<b>⚠️ Visual generation error:</b><br>{spec['error']}</div>"
            ))
            return

        # ── No visual needed ──────────────────────────────────────────────────
        if not spec.get("should_visualize", False) or spec.get("visual_type") == "none":
            display(HTML(
                "<div style='padding:12px;background:#f0fdf4;border:1px solid #86efac;"
                "border-radius:8px;color:#166534;'>"
                "<b>ℹ️ No visual needed for this section.</b><br>"
                "The text explanation is already the clearest representation."
                "</div>"
            ))
            return

        title    = spec.get("title", "")
        vtype    = spec.get("visual_type", "")
        why      = spec.get("why_helpful", "")
        desc     = spec.get("description", "")

        # Header
        type_badge_colors = {
            "flowchart":   "#dbeafe", "timeline":    "#fef3c7",
            "process":     "#dcfce7", "graph":       "#f3e8ff",
            "concept_map": "#fff1f2",
        }
        badge_color = type_badge_colors.get(vtype, "#f1f5f9")
        display(HTML(
            f"<div style='margin-bottom:8px;'>"
            f"<span style='font-size:1.15em;font-weight:bold;color:#1e3a5f;'>{title}</span>&nbsp;"
            f"<span style='background:{badge_color};padding:2px 10px;border-radius:12px;"
            f"font-size:0.85em;color:#374151;font-weight:600;'>{vtype}</span>"
            f"</div>"
            f"<p style='color:#4b5563;font-size:0.93em;margin:2px 0 10px 0;'>{desc}</p>"
        ))

        # Render visual
        img_b64 = render_visual_spec(spec, profile)
        if img_b64:
            display(HTML(
                f"<img src='data:image/png;base64,{img_b64}' "
                f"style='max-width:100%;border:1px solid #e5e7eb;"
                f"border-radius:10px;padding:6px;background:#fff;' "
                f"alt='{title}' />"
            ))
        else:
            # Rendering returned nothing — show a plain-text fallback
            nodes  = spec.get("nodes", [])
            edges  = spec.get("edges", [])
            labels = spec.get("labels", [])
            if vtype == "timeline":
                chain = " → ".join(nodes)
                display(HTML(
                    f"<div style='font-size:1.1em;padding:12px;background:#fef9c3;"
                    f"border-radius:8px;letter-spacing:0.03em;'>{chain}</div>"
                ))
            else:
                items = "".join(
                    f"<div style='margin:6px 0;padding:8px 14px;background:#f8fafc;"
                    f"border-left:4px solid #6366f1;border-radius:4px;"
                    f"font-weight:600;'>{n}</div>" +
                    (f"<div style='text-align:center;color:#6b7280;font-size:1.3em;'>↓</div>"
                     if i < len(nodes) - 1 else "")
                    for i, n in enumerate(nodes)
                )
                display(HTML(f"<div style='max-width:400px;'>{items}</div>"))

        # Why this visual?
        if why:
            display(HTML(
                f"<div style='margin-top:10px;padding:10px 14px;"
                f"background:#f0f9ff;border-left:4px solid #0ea5e9;"
                f"border-radius:6px;color:#0c4a6e;font-size:0.92em;'>"
                f"<b>Why this visual?</b> {why}"
                f"</div>"
            ))

visual_generate_button.on_click(on_generate_visual)

# ── ASSEMBLE FULL UI ───────────────────────────────────────────────────────────
voice_divider   = widgets.HTML("<hr style='margin:20px 0 12px 0;border:none;border-top:2px solid #ccc;'>")
voice_label     = widgets.HTML("<b style='font-size:1.05em;'>Need help understanding the lesson?</b>")
voice_status    = widgets.HTML(
    "<div id='prism-voice-status' style='color:#0284c7;font-weight:500;font-size:0.95em;margin:4px 0 6px 0;'>"
    "Press 🎙️ Speak or type a question below."
    "</div>"
)
voice_btns      = widgets.HBox([mic_button, ask_button], layout=widgets.Layout(gap="10px", margin="8px 0"))
read_aloud_row  = widgets.HBox([read_aloud_button], layout=widgets.Layout(margin="6px 0"))

visual_divider  = widgets.HTML("<hr style='margin:20px 0 12px 0;border:none;border-top:2px solid #ccc;'>")
visual_label    = widgets.HTML(
    "<b style='font-size:1.05em;'>📊 Visual Explanation</b>"
    "<span style='color:#6b7280;font-size:0.88em;margin-left:10px;'>"
    "Generates a diagram grounded in the current lesson and profile.</span>"
)

full_ui = widgets.VBox([
    # --- existing accessibility UI ---
    profile_selector,
    text_input,
    generate_button,
    transformation_output,
    # --- voice section ---
    voice_divider,
    voice_label,
    voice_status,
    voice_btns,
    voice_text_input,
    _voice_bridge,
    voice_output,
    read_aloud_row,
    # --- visual section ---
    visual_divider,
    visual_label,
    visual_generate_button,
    visual_output,
])

display(full_ui)
display(Javascript(VOICE_JS_CODE))

<IPython.core.display.Javascript object>

## 4. QUIZ GENERATION

Each chunk gets one LLM call. The parser accepts plain JSON and JSON wrapped in Markdown code fences.

In [5]:
QUIZ_PROMPT = """Create one practice question based only on the chunk below. Adjust phrasing complexity to the learner profile: {profile}. Return valid JSON only with exactly these keys: question, options, answer, explanation. The options value must be an array of exactly 4 strings. The answer must exactly match one option. Do not use information outside the chunk. Include the marker QUIZ_JSON nowhere except in this instruction context.

CHUNK:
{chunk}"""

def generate_quiz(chunk: str, profile: str) -> Dict[str, Any]:
    """Generate and validate one profile-aware quiz question for a text chunk."""
    required_keys = {"question", "options", "answer", "explanation"}
    quiz = {}
    for _ in range(2):
        response = call_llm(QUIZ_PROMPT.format(chunk=chunk, profile=profile))
        try:
            quiz = _parse_json_response(response)
        except Exception:
            continue
        if not isinstance(quiz, dict) or not required_keys.issubset(set(quiz)):
            continue
        opts = quiz.get("options")
        if not isinstance(opts, list):
            continue
        if len(opts) > 4 and quiz.get("answer") in opts:
            others = [o for o in opts if o != quiz["answer"]]
            quiz["options"] = [quiz["answer"]] + others[:3]
        if len(quiz.get("options", [])) == 4 and quiz.get("answer") in quiz["options"]:
            return {k: quiz[k] for k in required_keys}

    if set(quiz) != required_keys or not isinstance(quiz.get("options"), list) or len(quiz.get("options", [])) != 4:
        raise ValueError("Quiz response must contain question, four options, answer, and explanation.")
    if quiz["answer"] not in quiz["options"]:
        raise ValueError("Quiz answer must match one of the options.")
    return quiz

cognitive_chunks = transform_text(SOURCE_TEXT, "cognitive_load")["chunks"]
for index, chunk in enumerate(cognitive_chunks[:3], start=1):
    print(f"\n--- QUIZ FOR CHUNK {index} ---")
    print(json.dumps(generate_quiz(chunk, "cognitive_load"), indent=2))


--- QUIZ FOR CHUNK 1 ---


{
  "explanation": "Photosynthesis takes place in the chloroplasts, which contain chlorophyll and other components necessary for converting light energy into chemical energy.",
  "question": "What is the primary organelle where photosynthesis occurs in plant cells?",
  "answer": "Chloroplasts",
  "options": [
    "Mitochondria",
    "Chloroplasts",
    "Nucleus",
    "Ribosome"
  ]
}

--- QUIZ FOR CHUNK 2 ---


{
  "explanation": "Chlorophyll absorbs light most efficiently in the blue and red wavelengths of the visible spectrum, which is why plants appear green to the human eye.",
  "question": "Which wavelengths of light does chlorophyll absorb most efficiently?",
  "answer": "Blue and red",
  "options": [
    "Blue and red",
    "Green and yellow",
    "Blue and green",
    "Red and yellow"
  ]
}

--- QUIZ FOR CHUNK 3 ---


{
  "explanation": "The light-dependent reactions occur in the thylakoid membranes, where sunlight is used to split water molecules, releasing oxygen and producing the energy-carrying molecules ATP and NADPH.",
  "question": "What is the primary function of the light-dependent reactions in photosynthesis?",
  "answer": "To split water and generate ATP and NADPH",
  "options": [
    "To produce glucose directly",
    "To split water and generate ATP and NADPH",
    "To fix carbon dioxide into sugars",
    "To transport electrons to the stroma"
  ]
}


## 5. END-TO-END TEST

This final cell runs raw text through all three profile paths and generates practice questions for the transformed content.

In [6]:
def chunks_for_quiz(transformed: Dict[str, Any]) -> List[str]:
    """Normalize a transformed profile result into quiz-ready chunks."""
    if transformed["profile"] == "cognitive_load":
        return transformed["chunks"]
    return [transformed["text"]]

pipeline_summary: Dict[str, Any] = {}
for profile in ("dyslexia", "low_vision", "cognitive_load"):
    transformed  = transform_text(SOURCE_TEXT, profile)
    quiz_results = [generate_quiz(chunk, profile) for chunk in chunks_for_quiz(transformed)[:3]]
    pipeline_summary[profile] = {
        "transformed_keys": list(transformed.keys()),
        "quiz_count":        len(quiz_results),
        "first_question":    quiz_results[0]["question"] if quiz_results else None,
    }

print("\n=== FINAL PIPELINE SUMMARY ===")
print(json.dumps(pipeline_summary, indent=2))


=== FINAL PIPELINE SUMMARY ===
{
  "dyslexia": {
    "transformed_keys": [
      "profile",
      "text"
    ],
    "quiz_count": 1,
    "first_question": "Which part of the chloroplast is where the light\u2011dependent reactions of photosynthesis take place?"
  },
  "low_vision": {
    "transformed_keys": [
      "profile",
      "text",
      "formatting"
    ],
    "quiz_count": 1,
    "first_question": "Which of the following is produced during the light-dependent reactions of photosynthesis?"
  },
  "cognitive_load": {
    "transformed_keys": [
      "profile",
      "chunks"
    ],
    "quiz_count": 3,
    "first_question": "Which of the following best describes the overall chemical equation for photosynthesis as presented in the chunk?"
  }
}
